# Imports

In [2]:
# import muon
import numpy as np
import mudata as md
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import davies_bouldin_score
from sklearn.neighbors import NearestNeighbors

# Read Data

Working off of the preprocessed tonsil h5mu

In [3]:
# mdata = md.read_h5mu("../../data/tonsil/tonsil_embedded.h5mu")
mdata = md.read_h5mu("../../data/tonsil/tonsil_embedded_tmp_svg.h5mu")
rna_adata = mdata.mod['RNA']
prot_adata = mdata.mod['Protein']

/opt/anaconda3/lib/python3.13/site-packages/mudata/_core/mudata.py:1598: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/opt/anaconda3/lib/python3.13/site-packages/mudata/_core/mudata.py:1461: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


In [4]:
rna_moran = sc.metrics.morans_i(rna_adata, obsm="X_spatial_totalVI")

In [5]:
pd.DataFrame(data = {'Moran_I': rna_moran}).to_csv('../results/Moran_I_our_model_metrics.csv', index=False)

# cLISI

In [6]:
def compute_lisi(emb, labels, perplexity=30):
    n_neighbors = int(3 * perplexity)
    nn = NearestNeighbors(n_neighbors=n_neighbors + 1).fit(emb)
    distances, indices = nn.kneighbors(emb)
    indices = indices[:, 1:]

    labels = np.array(labels)
    out = []

    for i in range(emb.shape[0]):
        neigh = labels[indices[i]]
        p = pd.value_counts(neigh) / len(neigh)
        entropy = -(p * np.log(p)).sum()
        out.append(np.exp(entropy))

    return np.array(out)

In [7]:
rna_adata.obs["cLISI_kmeans5"] = compute_lisi(rna_adata.obsm["X_spatial_totalVI"], rna_adata.obs["kmeans5"])
rna_adata.obs["cLISI_louvain"] = compute_lisi(rna_adata.obsm["X_spatial_totalVI"], rna_adata.obs["louvain_eval"])
rna_adata.obs["cLISI_leiden"] = compute_lisi(rna_adata.obsm["X_spatial_totalVI"], rna_adata.obs["leiden_eval"])

/var/folders/td/snqs86gn12g5nxg94yl2nhrr0000gn/T/ipykernel_46227/3353607529.py:12: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  p = pd.value_counts(neigh) / len(neigh)
/var/folders/td/snqs86gn12g5nxg94yl2nhrr0000gn/T/ipykernel_46227/3353607529.py:12: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  p = pd.value_counts(neigh) / len(neigh)
/var/folders/td/snqs86gn12g5nxg94yl2nhrr0000gn/T/ipykernel_46227/3353607529.py:12: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  p = pd.value_counts(neigh) / len(neigh)
/var/folders/td/snqs86gn12g5nxg94yl2nhrr0000gn/T/ipykernel_46227/3353607529.py:12: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  p = pd.v

In [8]:
rna_adata.obs[['cLISI_kmeans5', 'cLISI_louvain', 'cLISI_leiden']].to_csv("../results/LISI_metrics_our_model_totalVI.csv")

# DBI

In [9]:
# Extract embedding and labels
X = rna_adata.obsm["X_spatial_totalVI"]
labels = rna_adata.obs["leiden_eval"].astype(int).values

In [10]:
dbi_list = []

# compute DBI for each single PC (1D DBI)
for i in range(X.shape[1]):
    Xi = X[:, i].reshape(-1, 1)
    dbi_list.append(davies_bouldin_score(Xi, labels))

dbi_list = np.array(dbi_list)
print(dbi_list)

[27.83430842 16.53366406 27.78058172  7.81742998 11.86006664  8.70991808
 10.06634626 19.83122844  7.45135232 14.3922041  13.33693394 33.13735437
 34.48309826 28.63240825 21.42763294 25.73220144 11.46399977 11.08537521
 11.97669934  6.01085689]


In [11]:
pd.DataFrame(data = {"DBI_leiden": dbi_list}).to_csv("../results/DBI_our_model_metrics.csv", index = False)